# RQ5 - Sensitivity to Evaluation Metrics

**Research Question:** How does the ranking of candidate machine learning models change when evaluated using different classification metrics?

This notebook loads the raw dataset and saves the actual result table as CSV and the actual figure as PDF.

In [ ]:

# Predictive Maintenance ML Assignment - Common Setup
# This notebook starts from the raw dataset file and generates the actual table/figure for this research question.

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

# ========== USER SETTINGS ==========
# Works with CSV and Excel. Update this path if you run the notebook on Kaggle or your own machine.
DATA_PATH = "../data/predictive_maintenance_cleaned_New.csv"

# If the notebook is run from another working directory, also try /mnt/data.
if not os.path.exists(DATA_PATH):
    alt_path = "/mnt/data/predictive_maintenance_cleaned_New.csv"
    if os.path.exists(alt_path):
        DATA_PATH = alt_path

RESULTS_TABLE_DIR = "../results/tables"
RESULTS_FIGURE_DIR = "../results/figures"
os.makedirs(RESULTS_TABLE_DIR, exist_ok=True)
os.makedirs(RESULTS_FIGURE_DIR, exist_ok=True)

TARGET = "failure_within_24h"
LEAKAGE_COLUMNS = ["rul_hours", "failure_type", "estimated_repair_cost"]
RANDOM_STATE = 42

def load_dataset(path=DATA_PATH):
    """Load CSV or Excel dataset."""
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    return pd.read_csv(path)

def add_time_features(df):
    """Create time-based features from timestamp if available."""
    df = df.copy()
    if "timestamp" in df.columns:
        ts = pd.to_datetime(df["timestamp"], errors="coerce")
        df["hour"] = ts.dt.hour
        df["day_of_week"] = ts.dt.dayofweek
        df["month"] = ts.dt.month
        df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
        df = df.drop(columns=["timestamp"])
    return df

def prepare_xy(df, use_leakage=False, include_machine_id=True, include_time_features=True):
    """Prepare feature matrix X and target y."""
    df = df.copy()
    if include_time_features:
        df = add_time_features(df)
    else:
        if "timestamp" in df.columns:
            df = df.drop(columns=["timestamp"])

    if TARGET not in df.columns:
        raise ValueError(f"Target column '{TARGET}' not found in dataset.")

    drop_cols = [TARGET]
    if not use_leakage:
        drop_cols += [c for c in LEAKAGE_COLUMNS if c in df.columns]
    if not include_machine_id and "machine_id" in df.columns:
        drop_cols.append("machine_id")

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df[TARGET].astype(int)
    return X, y

def split_data(X, y, test_size=0.2):
    return train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=RANDOM_STATE
    )

def get_feature_types(X):
    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical_features = [c for c in X.columns if c not in categorical_features]
    return numerical_features, categorical_features

def make_preprocessor(X, scale_numeric=True):
    numerical_features, categorical_features = get_feature_types(X)

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)

    try:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot)
    ])

    transformers = []
    if numerical_features:
        transformers.append(("num", numeric_transformer, numerical_features))
    if categorical_features:
        transformers.append(("cat", categorical_transformer, categorical_features))

    return ColumnTransformer(transformers=transformers, remainder="drop")

def get_model(name):
    """Return a classification model by name."""
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    if name == "Decision Tree":
        return DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=RANDOM_STATE)
    if name == "k-NN":
        return KNeighborsClassifier(n_neighbors=7)
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=250, max_depth=None, min_samples_leaf=2,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        )
    if name == "SVM":
        return SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE)
    if name == "XGBoost":
        if XGBOOST_AVAILABLE:
            return XGBClassifier(
                n_estimators=250, learning_rate=0.05, max_depth=4,
                subsample=0.9, colsample_bytree=0.9,
                eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
            )
        return GradientBoostingClassifier(random_state=RANDOM_STATE)
    if name == "Gradient Boosting":
        return GradientBoostingClassifier(random_state=RANDOM_STATE)
    raise ValueError(f"Unknown model name: {name}")

def build_pipeline(model_name, X, scale_numeric=True):
    return Pipeline(steps=[
        ("preprocessor", make_preprocessor(X, scale_numeric=scale_numeric)),
        ("model", get_model(model_name))
    ])

def predict_scores(model, X_test):
    """Return probability score for positive class where possible."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_test)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        # Min-max scale decision scores to [0, 1] for AUC compatibility only
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    return None

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_score = predict_scores(model, X_test)
    auc = roc_auc_score(y_test, y_score) if y_score is not None and len(np.unique(y_test)) > 1 else np.nan
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "AUC": auc
    }

def format_metric_table(df, metric_cols=None):
    out = df.copy()
    if metric_cols is None:
        metric_cols = [c for c in ["Accuracy", "Precision", "Recall", "F1-score", "AUC"] if c in out.columns]
    for col in metric_cols:
        out[col] = out[col].astype(float).round(4)
    return out

def save_table(df, filename):
    path = os.path.join(RESULTS_TABLE_DIR, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path

def save_figure(filename):
    path = os.path.join(RESULTS_FIGURE_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved figure: {path}")
    plt.show()
    return path

def add_dummy_note():
    plt.figtext(0.5, -0.02, "Actual results generated from the dataset.", ha="center", fontsize=9, style="italic")


In [ ]:

# RQ5: Sensitivity to Evaluation Metrics

df = load_dataset()
X, y = prepare_xy(df, use_leakage=False, include_machine_id=True, include_time_features=True)
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2)

candidate_models = ["Logistic Regression", "Decision Tree", "k-NN", "Random Forest", "SVM", "XGBoost"]
rows = []

for model_name in candidate_models:
    scale = model_name in ["Logistic Regression", "k-NN", "SVM"]
    model = build_pipeline(model_name, X_train, scale_numeric=scale)
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test)
    metrics["Model"] = model_name if not (model_name == "XGBoost" and not XGBOOST_AVAILABLE) else "Gradient Boosting (fallback)"
    rows.append(metrics)

performance = pd.DataFrame(rows)[["Model", "Accuracy", "Precision", "Recall", "F1-score", "AUC"]]
performance = format_metric_table(performance)

rank_df = pd.DataFrame({"Model": performance["Model"]})
for metric in ["Accuracy", "Precision", "Recall", "F1-score", "AUC"]:
    rank_df[f"Rank by {metric}"] = performance[metric].rank(ascending=False, method="min").astype(int)

display(rank_df)
save_table(rank_df, "RQ5_metric_sensitivity.csv")

# Figure 5: bump chart
rank_long = rank_df.melt(id_vars="Model", var_name="Metric", value_name="Rank")
rank_long["Metric"] = rank_long["Metric"].str.replace("Rank by ", "", regex=False)
metric_order = ["Accuracy", "Precision", "Recall", "F1-score", "AUC"]
rank_long["Metric"] = pd.Categorical(rank_long["Metric"], categories=metric_order, ordered=True)
rank_long = rank_long.sort_values(["Model", "Metric"])

plt.figure(figsize=(11, 6))
for model_name, group in rank_long.groupby("Model"):
    plt.plot(group["Metric"], group["Rank"], marker="o", linewidth=2, label=model_name)
    for _, row in group.iterrows():
        plt.text(row["Metric"], row["Rank"], str(row["Rank"]), ha="center", va="center", fontsize=8, color="white",
                 bbox=dict(boxstyle="circle,pad=0.25", fc="black", ec="none", alpha=0.75))

plt.gca().invert_yaxis()
plt.yticks(range(1, len(candidate_models)+1))
plt.title("Figure 5. Variation in Model Ranking Across Evaluation Metrics", fontsize=14, weight="bold")
plt.xlabel("Evaluation Metric")
plt.ylabel("Rank")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="lower center", bbox_to_anchor=(0.5, -0.30), ncol=3)
add_dummy_note()
save_figure("Figure_5_metric_ranking_variation.pdf")
